​Verify GPU Access in Code

In [1]:

import torch

if torch.cuda.is_available():
    device = torch.device("cuda")
    print(f"GPU active: {torch.cuda.get_device_name(0)}")
    print(f"Total VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.2f} GB")
else:
    print("WARNING: GPU not detected! Make sure you selected GPU in Runtime settings.")
    device = torch.device("cpu")

GPU active: Tesla T4
Total VRAM: 15.64 GB


## **Data Preparation Pipeline**


In [2]:

import math
import torch
from torch.utils.data import Dataset, DataLoader
from datasets import load_dataset
from transformers import AutoTokenizer

class TokenStreamDataset(Dataset):
    """
    Tokenizes a text stream and constructs contiguous sequence blocks of length (seq_len + 1).
    Input:  tokens[0 : seq_len]
    Target: tokens[1 : seq_len + 1]
    """
    def __init__(self, dataset_name: str = "roneneldan/TinyStories",
                 tokenizer_name: str = "gpt2",
                 seq_len: int = 256,
                 target_total_tokens: int = 50_000_000,
                 split: str = "train"):

        self.seq_len = seq_len
        self.target_tokens = target_total_tokens

        print(f"Loading dataset '{dataset_name}' [{split}]...")
        dataset = load_dataset(dataset_name, split=split, streaming=True)

        print(f"Loading tokenizer '{tokenizer_name}'...")
        tokenizer = AutoTokenizer.from_pretrained(tokenizer_name)
        if tokenizer.pad_token is None:
            tokenizer.pad_token = tokenizer.eos_token

        print(f"Streaming and tokenizing text until ~{target_total_tokens:,} tokens are reached...")

        all_tokens = []
        token_count = 0

        for item in dataset:
            text = item.get("text", "")
            if not text.strip():
                continue

            # Encode without adding special tokens manually (GPT-2 handles EOS via text boundary)
            ids = tokenizer.encode(text) + [tokenizer.eos_token_id]
            all_tokens.extend(ids)
            token_count += len(ids)

            if token_count >= target_total_tokens:
                break

        print(f"Total tokens collected: {token_count:,}")

        # Trim to exact total token count desired and cast to tensor
        all_tokens = all_tokens[:target_total_tokens]
        self.tokens = torch.tensor(all_tokens, dtype=torch.long)

        # Calculate number of complete chunks of length (seq_len + 1)
        self.chunk_size = seq_len + 1
        self.num_samples = len(self.tokens) // self.chunk_size

        # Trim unused tail tokens
        self.tokens = self.tokens[: self.num_samples * self.chunk_size]
        print(f"Dataset formatted into {self.num_samples:,} contiguous samples of length {seq_len}.")

    def __len__(self):
        return self.num_samples

    def __getitem__(self, idx):
        start_idx = idx * self.chunk_size
        chunk = self.tokens[start_idx : start_idx + self.chunk_size]

        # Autoregressive shifting
        x = chunk[:-1]
        y = chunk[1:]
        return x, y

def get_dataloader_and_steps(batch_size: int,
                             seq_len: int = 256,
                             target_total_tokens: int = 50_000_000):
    """
    Helper function to instantiate DataLoader and calculate exact gradient steps needed.
    """
    dataset = TokenStreamDataset(
        dataset_name="roneneldan/TinyStories",
        tokenizer_name="gpt2",
        seq_len=seq_len,
        target_total_tokens=target_total_tokens
    )

    loader = DataLoader(
        dataset,
        batch_size=batch_size,
        shuffle=True,
        drop_last=True,
        pin_memory=True if torch.cuda.is_available() else False
    )

    tokens_per_step = batch_size * seq_len
    total_steps = len(loader)
    total_tokens_processed = total_steps * tokens_per_step

    print("\n--- Pipeline Summary ---")
    print(f"Sequence Length (S)   : {seq_len}")
    print(f"Batch Size (B)        : {batch_size}")
    print(f"Tokens Per Step (B*S) : {tokens_per_step:,}")
    print(f"Total Steps           : {total_steps:,}")
    print(f"Total Tokens Trained  : {total_tokens_processed:,}")
    print("------------------------\n")

    return loader, total_steps

Quick Sanity Test

In [3]:
# Test creating a DataLoader with a baseline batch size of 32
train_loader, steps = get_dataloader_and_steps(batch_size=32, seq_len=256, target_total_tokens=50_000_000)

# Inspect first batch shape
x_batch, y_batch = next(iter(train_loader))
print(f"Input batch shape  (B, S): {x_batch.shape}")
print(f"Target batch shape (B, S): {y_batch.shape}")

Loading dataset 'roneneldan/TinyStories' [train]...


README.md:   0%|          | 0.00/1.06k [00:00<?, ?B/s]

Loading tokenizer 'gpt2'...


config.json:   0%|          | 0.00/665 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/26.0 [00:00<?, ?B/s]

...

merges.txt:   0%|          | 0.00/456k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/1.36M [00:00<?, ?B/s]

Streaming and tokenizing text until ~50,000,000 tokens are reached...


[transformers] Token indices sequence length is longer than the specified maximum sequence length for this model (1106 > 1024). Running this sequence through the model will result in indexing errors


Total tokens collected: 50,000,121
Dataset formatted into 194,552 contiguous samples of length 256.

--- Pipeline Summary ---
Sequence Length (S)   : 256
Batch Size (B)        : 32
Tokens Per Step (B*S) : 8,192
Total Steps           : 6,079
Total Tokens Trained  : 49,799,168
------------------------

Input batch shape  (B, S): torch.Size([32, 256])
Target batch shape (B, S): torch.Size([32, 256])


## Non-reversible baseline architecture, along with a baseline training loop that tracks Loss, Tokens/Sec, and Peak VRAM (torch.cuda.max_memory_allocated()):

In [7]:

import time
import math
import torch
import torch.nn as nn
import torch.nn.functional as F

# ==========================================
# 1. Standard Transformer Components
# ==========================================

class CausalSelfAttention(nn.Module):
    def __init__(self, d_model=384, n_head=6, dropout=0.0):
        super().__init__()
        assert d_model % n_head == 0
        self.n_head = n_head
        self.head_dim = d_model // n_head

        self.c_attn = nn.Linear(d_model, 3 * d_model)
        self.c_proj = nn.Linear(d_model, d_model)
        self.dropout = dropout

    def forward(self, x):
        B, S, C = x.size()
        q, k, v = self.c_attn(x).chunk(3, dim=-1)

        q = q.view(B, S, self.n_head, self.head_dim).transpose(1, 2)
        k = k.view(B, S, self.n_head, self.head_dim).transpose(1, 2)
        v = v.view(B, S, self.n_head, self.head_dim).transpose(1, 2)

        # PyTorch FlashAttention / SDPA for fast forward/backward
        y = F.scaled_dot_product_attention(
            q, k, v,
            is_causal=True,
            dropout_p=self.dropout if self.training else 0.0
        )

        y = y.transpose(1, 2).contiguous().view(B, S, C)
        return self.c_proj(y)

class MLP(nn.Module):
    def __init__(self, d_model=384, d_ff=1536):
        super().__init__()
        self.c_fc = nn.Linear(d_model, d_ff)
        self.gelu = nn.GELU()
        self.c_proj = nn.Linear(d_ff, d_model)

    def forward(self, x):
        return self.c_proj(self.gelu(self.c_fc(x)))

class StandardTransformerBlock(nn.Module):
    def __init__(self, d_model=384, n_head=6):
        super().__init__()
        self.ln_1 = nn.LayerNorm(d_model)
        self.attn = CausalSelfAttention(d_model=d_model, n_head=n_head)
        self.ln_2 = nn.LayerNorm(d_model)
        self.mlp = MLP(d_model=d_model, d_ff=4 * d_model)

    def forward(self, x):
        # Standard Residual Connections: store activations for backprop
        x = x + self.attn(self.ln_1(x))
        x = x + self.mlp(self.ln_2(x))
        return x

class StandardLM(nn.Module):
    def __init__(self, vocab_size=50257, d_model=384, n_layer=6, n_head=6, max_seq_len=256):
        super().__init__()
        self.tok_emb = nn.Embedding(vocab_size, d_model)
        self.pos_emb = nn.Embedding(max_seq_len, d_model)
        self.blocks = nn.ModuleList([
            StandardTransformerBlock(d_model=d_model, n_head=n_head) for _ in range(n_layer)
        ])
        self.ln_f = nn.LayerNorm(d_model)
        self.head = nn.Linear(d_model, vocab_size, bias=False)

        # Weight tying (optional, standard for GPT models)
        self.tok_emb.weight = self.head.weight

    def forward(self, idx):
        B, S = idx.size()
        pos = torch.arange(0, S, dtype=torch.long, device=idx.device)
        x = self.tok_emb(idx) + self.pos_emb(pos)

        for block in self.blocks:
            x = block(x)

        x = self.ln_f(x)
        logits = self.head(x)
        return logits

    def get_num_params(self):
        return sum(p.numel() for p in self.parameters())

# ==========================================
# 2. Baseline Training & Profiling Function
# ==========================================

def train_baseline_model(model, train_loader, epochs_or_steps, device, lr=3e-4):
    model.to(device)
    optimizer = torch.optim.AdamW(model.parameters(), lr=lr)
    criterion = nn.CrossEntropyLoss()

    # Reset peak memory stats before starting training
    torch.cuda.reset_peak_memory_stats(device)
    model.train()

    start_time = time.time()
    total_tokens_processed = 0

    print(f"Starting Baseline Training...")
    print(f"Model Parameters: {model.get_num_params():,}")

    for step, (x_batch, y_batch) in enumerate(train_loader):
        x_batch = x_batch.to(device, non_blocking=True)
        y_batch = y_batch.to(device, non_blocking=True)

        optimizer.zero_grad(set_to_none=True)
        logits = model(x_batch)
        loss = criterion(logits.view(-1, logits.size(-1)), y_batch.view(-1))

        loss.backward()
        optimizer.step()

        batch_tokens = x_batch.numel()
        total_tokens_processed += batch_tokens

        if (step + 1) % 100 == 0 or step == 0:
            elapsed = time.time() - start_time
            tok_per_sec = total_tokens_processed / elapsed
            peak_vram_mb = torch.cuda.max_memory_allocated(device) / (1024 ** 2)
            print(f"Step [{step+1}/{len(train_loader)}] | Loss: {loss.item():.4f} | Throughput: {tok_per_sec:.0f} tok/s | Peak VRAM: {peak_vram_mb:.2f} MB")

    total_time = time.time() - start_time
    final_peak_vram_mb = torch.cuda.max_memory_allocated(device) / (1024 ** 2)
    avg_throughput = total_tokens_processed / total_time

    print("\n--- Baseline Run Summary ---")
    print(f"Final Loss       : {loss.item():.4f}")
    print(f"Avg Throughput   : {avg_throughput:.2f} tokens/sec")
    print(f"Peak VRAM        : {final_peak_vram_mb:.2f} MB")
    print("---------------------------\n")

    return loss.item(), avg_throughput, final_peak_vram_mb

In [5]:
import torch
import torch.nn as nn
from torch.autograd import Function

# ==========================================
# 1. Euler Reversible Autograd Function
# ==========================================

class EulerReversibleFunction(Function):
    @staticmethod
    def forward(ctx, x1, x2, F_module, G_module, *weights):
        # Forward updates:
        # y1 = x1 + F(x2)
        # y2 = x2 + G(y1)
        with torch.no_grad():
            f_out = F_module(x2)
            y1 = x1 + f_out
            g_out = G_module(y1)
            y2 = x2 + g_out

        # Store modules and outputs for backward reconstruction
        ctx.F_module = F_module
        ctx.G_module = G_module
        # Save ONLY final outputs (O(1) memory per layer)
        ctx.save_for_backward(y1, y2)
        return y1, y2

    @staticmethod
    def backward(ctx, dy1, dy2):
        y1, y2 = ctx.saved_tensors
        F_module = ctx.F_module
        G_module = ctx.G_module

        # ----------------------------------------------------
        # 1. Reverse Pass: Reconstruct (x1, x2) from (y1, y2)
        # ----------------------------------------------------
        with torch.no_grad():
            # x2 = y2 - G(y1)
            g_out = G_module(y1)
            x2 = y2 - g_out

            # x1 = y1 - F(x2)
            f_out = F_module(x2)
            x1 = y1 - f_out

        # ----------------------------------------------------
        # 2. Gradient Computation via Vector-Jacobian Product
        # ----------------------------------------------------
        # Detach inputs to record local computational sub-graphs
        x2_detach = x2.detach().requires_grad_(True)
        y1_detach = y1.detach().requires_grad_(True)

        with torch.enable_grad():
            f_eval = F_module(x2_detach)
            g_eval = G_module(y1_detach)

        # Collect sub-module parameters
        f_params = list(F_module.parameters())
        g_params = list(G_module.parameters())

        # Backprop through G(y1)
        g_grads = torch.autograd.grad(
            outputs=g_eval,
            inputs=[y1_detach] + g_params,
            grad_outputs=dy2,
            retain_graph=False
        )
        dy1_from_g = g_grads[0]
        g_param_grads = g_grads[1:]

        # Accumulate stream gradient for y1: dy1_total = dy1 + dG/dy1
        dy1_total = dy1 + dy1_from_g

        # Backprop through F(x2)
        f_grads = torch.autograd.grad(
            outputs=f_eval,
            inputs=[x2_detach] + f_params,
            grad_outputs=dy1_total,
            retain_graph=False
        )
        dx2_from_f = f_grads[0]
        f_param_grads = f_grads[1:]

        # dx1 = dy1_total
        dx1 = dy1_total
        # dx2 = dy2 + dF/dx2
        dx2 = dy2 + dx2_from_f

        # Accumulate gradients into module parameter .grad attributes
        for p, g in zip(f_params, f_param_grads):
            if p.grad is None:
                p.grad = g
            else:
                p.grad += g

        for p, g in zip(g_params, g_param_grads):
            if p.grad is None:
                p.grad = g
            else:
                p.grad += g

        # Return gradients matching input signature: (x1, x2, F_module, G_module, *weights)
        return dx1, dx2, None, None, *[None for _ in f_params + g_params]


# ==========================================
# 2. Midpoint Reversible Autograd Function
# ==========================================

class MidpointReversibleFunction(Function):
    @staticmethod
    def forward(ctx, x1, x2, F_module, G_module, *weights):
        # Midpoint Formulation:
        # y1 = x1 + F(x2)
        # y2 = x2 + G(0.5 * (x1 + y1))
        with torch.no_grad():
            f_out = F_module(x2)
            y1 = x1 + f_out
            midpoint = 0.5 * (x1 + y1)
            g_out = G_module(midpoint)
            y2 = x2 + g_out

        ctx.F_module = F_module
        ctx.G_module = G_module
        # Save x1 and y1 for midpoint reconstruction
        ctx.save_for_backward(x1, y1, y2)
        return y1, y2

    @staticmethod
    def backward(ctx, dy1, dy2):
        x1, y1, y2 = ctx.saved_tensors
        F_module = ctx.F_module
        G_module = ctx.G_module

        with torch.no_grad():
            midpoint = 0.5 * (x1 + y1)
            g_out = G_module(midpoint)
            x2 = y2 - g_out

        x2_detach = x2.detach().requires_grad_(True)
        mid_detach = midpoint.detach().requires_grad_(True)

        with torch.enable_grad():
            f_eval = F_module(x2_detach)
            g_eval = G_module(mid_detach)

        f_params = list(F_module.parameters())
        g_params = list(G_module.parameters())

        # Backprop through G
        g_grads = torch.autograd.grad(
            outputs=g_eval,
            inputs=[mid_detach] + g_params,
            grad_outputs=dy2,
            retain_graph=False
        )
        dmid = g_grads[0]
        g_param_grads = g_grads[1:]

        # Midpoint distribution to dy1 and dx1
        dy1_total = dy1 + 0.5 * dmid
        dx1_base = 0.5 * dmid

        # Backprop through F
        f_grads = torch.autograd.grad(
            outputs=f_eval,
            inputs=[x2_detach] + f_params,
            grad_outputs=dy1_total,
            retain_graph=False
        )
        dx2_from_f = f_grads[0]
        f_param_grads = f_grads[1:]

        dx1 = dx1_base + dy1_total
        dx2 = dy2 + dx2_from_f

        for p, g in zip(f_params, f_param_grads):
            if p.grad is None:
                p.grad = g
            else:
                p.grad += g

        for p, g in zip(g_params, g_param_grads):
            if p.grad is None:
                p.grad = g
            else:
                p.grad += g

        return dx1, dx2, None, None, *[None for _ in f_params + g_params]


# ==========================================
# 3. High-Level Reversible Block Wrappers
# ==========================================

class ReversibleTransformerBlock(nn.Module):
    def __init__(self, d_model=384, n_head=6, integrator="euler"):
        super().__init__()
        self.integrator = integrator.lower()

        # Stream size is d_model // 2
        half_dim = d_model // 2

        # F_module: LayerNorm + Multi-Head Attention on half-dimension
        self.F_module = nn.Sequential(
            nn.LayerNorm(half_dim),
            CausalSelfAttention(d_model=half_dim, n_head=n_head)
        )

        # G_module: LayerNorm + MLP on half-dimension
        self.G_module = nn.Sequential(
            nn.LayerNorm(half_dim),
            MLP(d_model=half_dim, d_ff=4 * half_dim)
        )

    def forward(self, x1, x2):
        f_params = list(self.F_module.parameters())
        g_params = list(self.G_module.parameters())
        all_params = f_params + g_params

        if self.integrator == "euler":
            return EulerReversibleFunction.apply(x1, x2, self.F_module, self.G_module, *all_params)
        elif self.integrator == "midpoint":
            return MidpointReversibleFunction.apply(x1, x2, self.F_module, self.G_module, *all_params)
        else:
            raise ValueError(f"Unknown integrator: {self.integrator}")

**Phase 4 OverviewReversibleLM: Wrapper model that handles channel splitting ($d_{\text{model}} \rightarrow \frac{d_{\text{model}}}{2} + \frac{d_{\text{model}}}{2}$), passes streams through the stack of reversible blocks, and merges them back for final logit projection.Unified Benchmarking Suite: Measures Peak VRAM Memory (MB), Throughput (Tokens/sec), and Final Training Loss under identical hardware and batch settings.**

In [6]:
import time
import math
import torch
import torch.nn as nn
import torch.nn.functional as F

# Assumes StandardLM, StandardTransformerBlock, CausalSelfAttention, MLP,
# and ReversibleTransformerBlock (Phase 3) are loaded.

# ==========================================
# 1. Complete Reversible Language Model
# ==========================================

class ReversibleLM(nn.Module):
    def __init__(self, vocab_size=50257, d_model=384, n_layer=6, n_head=6, max_seq_len=256, integrator="euler"):
        super().__init__()
        assert d_model % 2 == 0, "d_model must be divisible by 2 for dual-stream splitting"

        self.d_model = d_model
        self.tok_emb = nn.Embedding(vocab_size, d_model)
        self.pos_emb = nn.Embedding(max_seq_len, d_model)

        # Stack of Reversible Blocks
        self.blocks = nn.ModuleList([
            ReversibleTransformerBlock(d_model=d_model, n_head=n_head, integrator=integrator)
            for _ in range(n_layer)
        ])

        self.ln_f = nn.LayerNorm(d_model)
        self.head = nn.Linear(d_model, vocab_size, bias=False)

        # Weight tying
        self.tok_emb.weight = self.head.weight

    def forward(self, idx):
        B, S = idx.size()
        pos = torch.arange(0, S, dtype=torch.long, device=idx.device)

        # Initial embeddings: shape [B, S, d_model]
        x = self.tok_emb(idx) + self.pos_emb(pos)

        # Split into two equal streams along hidden dimension: [B, S, d_model / 2]
        x1, x2 = torch.chunk(x, 2, dim=-1)

        # Pass through reversible block stack
        for block in self.blocks:
            x1, x2 = block(x1, x2)

        # Concatenate streams back to full d_model size
        x = torch.cat([x1, x2], dim=-1)

        x = self.ln_f(x)
        logits = self.head(x)
        return logits

    def get_num_params(self):
        return sum(p.numel() for p in self.parameters())


# ==========================================
# 2. Universal Benchmarking & Profiling Runner
# ==========================================

def run_benchmark(model_name, model, train_loader, device, num_steps=200, lr=3e-4):
    model.to(device)
    optimizer = torch.optim.AdamW(model.parameters(), lr=lr)
    criterion = nn.CrossEntropyLoss()

    # Reset CUDA memory stats
    torch.cuda.empty_cache()
    torch.cuda.reset_peak_memory_stats(device)

    model.train()
    start_time = time.time()
    total_tokens_processed = 0

    print(f"--- Running Benchmark: {model_name} ---")
    print(f"Parameters: {model.get_num_params():,}")

    running_loss = 0.0

    for step, (x_batch, y_batch) in enumerate(train_loader):
        if step >= num_steps:
            break

        x_batch = x_batch.to(device, non_blocking=True)
        y_batch = y_batch.to(device, non_blocking=True)

        optimizer.zero_grad(set_to_none=True)
        logits = model(x_batch)
        loss = criterion(logits.view(-1, logits.size(-1)), y_batch.view(-1))

        loss.backward()
        optimizer.step()

        batch_tokens = x_batch.numel()
        total_tokens_processed += batch_tokens
        running_loss += loss.item()

        if (step + 1) % 50 == 0 or step == 0:
            elapsed = time.time() - start_time
            tok_per_sec = total_tokens_processed / elapsed
            peak_vram_mb = torch.cuda.max_memory_allocated(device) / (1024 ** 2)
            print(f"Step [{step+1}/{num_steps}] | Loss: {loss.item():.4f} | Throughput: {tok_per_sec:.0f} tok/s | Peak VRAM: {peak_vram_mb:.2f} MB")

    total_time = time.time() - start_time
    final_peak_vram_mb = torch.cuda.max_memory_allocated(device) / (1024 ** 2)
    avg_throughput = total_tokens_processed / total_time
    avg_loss = running_loss / min(num_steps, len(train_loader))

    return {
        "architecture": model_name,
        "loss": avg_loss,
        "throughput": avg_throughput,
        "peak_vram_mb": final_peak_vram_mb
    }


# ==========================================
# 3. Execution Suite Execution Entry Point
# ==========================================

def run_comparative_experiment(train_loader, device="cuda"):
    # Common hyperparameters matching ~20M parameter spec
    config = {
        "vocab_size": 50257,
        "d_model": 384,
        "n_layer": 6,
        "n_head": 6,
        "max_seq_len": 256
    }

    # Instantiate models
    models = {
        "Standard Transformer": StandardLM(**config),
        "Euler Reversible Transformer": ReversibleLM(**config, integrator="euler"),
        "Midpoint Reversible Transformer": ReversibleLM(**config, integrator="midpoint")
    }

    results = []

    for name, model in models.items():
        res = run_benchmark(name, model, train_loader, device=device, num_steps=200)
        results.append(res)
        print("\n")

    # Print Comparative Results Table
    print("=" * 65)
    print(f"{'Architecture':<32} | {'Peak VRAM (MB)':<14} | {'Tok/Sec':<10} | {'Loss':<6}")
    print("=" * 65)
    for r in results:
        print(f"{r['architecture']:<32} | {r['peak_vram_mb']:<14.2f} | {r['throughput']:<10.0f} | {r['loss']:<6.4f}")
    print("=" * 65)

### Run Comparative Experiment

In [8]:
run_comparative_experiment(train_loader, device=device)

--- Running Benchmark: Standard Transformer ---
Parameters: 30,044,544
Step [1/200] | Loss: 11.0027 | Throughput: 7409 tok/s | Peak VRAM: 7584.31 MB
Step [50/200] | Loss: 6.0923 | Throughput: 16230 tok/s | Peak VRAM: 7822.22 MB
Step [100/200] | Loss: 6.0270 | Throughput: 15966 tok/s | Peak VRAM: 7822.22 MB
Step [150/200] | Loss: 6.0016 | Throughput: 16044 tok/s | Peak VRAM: 7822.22 MB
Step [200/200] | Loss: 5.9952 | Throughput: 16154 tok/s | Peak VRAM: 7822.22 MB


--- Running Benchmark: Euler Reversible Transformer ---
Parameters: 22,066,944
Step [1/200] | Loss: 10.9785 | Throughput: 18365 tok/s | Peak VRAM: 6711.04 MB
Step [50/200] | Loss: 6.0917 | Throughput: 19096 tok/s | Peak VRAM: 6880.42 MB
Step [100/200] | Loss: 5.9409 | Throughput: 19054 tok/s | Peak VRAM: 6880.42 MB
Step [150/200] | Loss: 5.9841 | Throughput: 19067 tok/s | Peak VRAM: 6880.42 MB
Step [200/200] | Loss: 5.9734 | Throughput: 19097 tok/s | Peak VRAM: 6880.42 MB


--- Running Benchmark: Midpoint Reversible Transfor